# Part 8: Bootstrap Uncertainty Quantification & HMM Ensemble Drift Audit

This notebook runs the reproducible Part 8 Python runner in Colab. It does not modify cleaned data or Part 1-7 outputs. The main specification uses 2000 circular moving block bootstrap replications with 13-week blocks and an 8-variant HMM ensemble drift audit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part8_uncertainty_quantification' / 'run_part8_uncertainty_quantification.py').exists(), 'Part 8 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part8_uncertainty_quantification/requirements-part8.txt

## Path configuration

The first path in each pair is the canonical Drive output folder. The second path supports locally downloaded and re-uploaded zip outputs that preserve the `*_outputs/...` folder structure.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART1_RUN_DIR = choose_existing('outputs/part1_btc_macro_state/colab_part1_seed42', 'outputs/part1_btc_macro_state_outputs/part1_btc_macro_state/colab_part1_seed42')
PART2_RUN_DIR = choose_existing('outputs/part2_portfolio_risk_budget/colab_part2_seed42', 'outputs/part2_portfolio_risk_budget_outputs/part2_portfolio_risk_budget/colab_part2_seed42')
PART3_RUN_DIR = choose_existing('outputs/part3_btc_state_dependence/colab_part3_seed42', 'outputs/part3_btc_state_dependence_outputs/part3_btc_state_dependence/colab_part3_seed42')
PART4_RUN_DIR = choose_existing('outputs/part4_conditional_btc_allocation/colab_part4_seed42', 'outputs/part4_conditional_btc_allocation_outputs/part4_conditional_btc_allocation/colab_part4_seed42')
PART5_RUN_DIR = choose_existing('outputs/part5_implementability_rebalancing/colab_part5_seed42', 'outputs/part5_implementability_rebalancing_outputs/part5_implementability_rebalancing/colab_part5_seed42')
PART6_RUN_DIR = choose_existing('outputs/part6_robustness_analysis/colab_part6_seed42', 'outputs/part6_robustness_analysis_outputs/part6_robustness_analysis/colab_part6_seed42')
PART7_RUN_DIR = choose_existing('outputs/part7_realtime_probabilistic_regime_robustness/colab_part7_seed42', 'outputs/part7_realtime_probabilistic_regime_robustness_outputs/part7_realtime_probabilistic_regime_robustness/colab_part7_seed42')
OUTPUT_DIR = Path('outputs/part8_uncertainty_quantification')
RUN_ID = 'colab_part8_seed42'
SEED = 42
BOOTSTRAP_REPS = 2000
BLOCK_LENGTH = 13

required_paths = {
    'INPUT_DIR': INPUT_DIR,
    'PART1_RUN_DIR': PART1_RUN_DIR,
    'PART2_RUN_DIR': PART2_RUN_DIR,
    'PART3_RUN_DIR': PART3_RUN_DIR,
    'PART4_RUN_DIR': PART4_RUN_DIR,
    'PART5_RUN_DIR': PART5_RUN_DIR,
    'PART6_RUN_DIR': PART6_RUN_DIR,
    'PART7_RUN_DIR': PART7_RUN_DIR,
}
for name, path in required_paths.items():
    print(f'{name:15s}', path.exists(), path)
    assert path.exists(), f'{name} not found. Check the path configuration cell.'

## Run Part 8

This cell can take several minutes because it runs 2000 bootstrap replications and the HMM ensemble audit.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable,
    'experiments/part8_uncertainty_quantification/run_part8_uncertainty_quantification.py',
    '--input-dir', str(INPUT_DIR),
    '--part1-run-dir', str(PART1_RUN_DIR),
    '--part2-run-dir', str(PART2_RUN_DIR),
    '--part3-run-dir', str(PART3_RUN_DIR),
    '--part4-run-dir', str(PART4_RUN_DIR),
    '--part5-run-dir', str(PART5_RUN_DIR),
    '--part6-run-dir', str(PART6_RUN_DIR),
    '--part7-run-dir', str(PART7_RUN_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--run-id', RUN_ID,
    '--seed', str(SEED),
    '--bootstrap-reps', str(BOOTSTRAP_REPS),
    '--block-length', str(BLOCK_LENGTH),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
print('Run directory:', RUN_DIR)
print('Results:', sorted(p.name for p in RESULTS.iterdir()))
print('Figures:', sorted(p.name for p in FIGURES.iterdir()))

In [ ]:
display(pd.read_json(RESULTS / 'bootstrap_configuration.json', typ='series'))
display(pd.read_csv(RESULTS / 'small_sample_state_uncertainty_audit.csv'))
display(pd.read_csv(RESULTS / 'risk_budget_cap_exceedance_summary.csv'))
display(pd.read_csv(RESULTS / 'hmm_ensemble_state_agreement.csv'))

In [ ]:
display(pd.read_csv(RESULTS / 'state_conditioned_btc_bootstrap_ci.csv').head(12))
display(pd.read_csv(RESULTS / 'conditional_rule_bootstrap_ci.csv').head(12))
display(pd.read_csv(RESULTS / 'realtime_vs_expost_bootstrap_ci.csv').head(12))

In [ ]:
from IPython.display import Image, display

for name in [
    'state_btc_bootstrap_ci.png',
    'rule_performance_bootstrap_ci.png',
    'risk_contribution_bootstrap_ci.png',
    'realtime_vs_expost_bootstrap_ci.png',
    'hmm_ensemble_agreement.png',
    'small_sample_state_uncertainty.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)